In [ ]:
#openwebtext datasetinin yüklenmesi ve kontrol edilmesi bloğu
from datasets import load_dataset
dataset = load_dataset("Skylion007/openwebtext", split="train")
print(dataset[0]['text'][:500])

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 64 #eldeki cümleyi 64 erli işleyecek
batch_size = 8 #paralel olarak kaç batchi inceleyeceğiz
max_iters = 500
learning_rate = 2e-5
eval_iters = 100
n_embd = 384
n_head = 8
n_layer = 8
dropout = 0.2
# Pre-training (OpenWebText) yapıyorsanız 'ai_save.pt' yapın
# Fine-tuning (Alpaca) yapıyorsanız 'ai_save_alpaca.pt' yapın
save_path = 'ai_save.pt'

In [27]:
import tiktoken
import torch
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab 
encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: enc.decode(l)
text_data = " ".join([dataset[i]['text'] for i in range(10000)])
data = torch.tensor(encode(text_data), dtype=torch.long)

In [ ]:
#sohbet dataseti her zaman yükleme
import json
import torch
import tiktoken

enc = tiktoken.get_encoding("gpt2")
dosya_yolu = "alpaca_data.json"
with open(dosya_yolu, 'r', encoding='utf-8') as f:
    alpaca_raw = json.load(f)
def format_alpaca_prompt(sample):
    if sample.get("input", "").strip() != "":
        return f"### Talimat:\n{sample['instruction']}\n\n### Girdi:\n{sample['input']}\n\n### Cevap:\n{sample['output']}<|endoftext|>"
    else:
        return f"### Talimat:\n{sample['instruction']}\n\n### Cevap:\n{sample['output']}<|endoftext|>"
all_tokens = []
for sample in alpaca_raw:
    formatted_text = format_alpaca_prompt(sample)
    tokens = enc.encode(formatted_text, allowed_special={"<|endoftext|>"})
    all_tokens.extend(tokens)

chat_data = torch.tensor(all_tokens, dtype=torch.long)
print(f"Sohbet eğitimine hazır toplam token sayısı: {len(chat_data)}")
n_chat = int(0.9 * len(chat_data))
train_data = chat_data[:n_chat]
val_data = chat_data[n_chat:]

print(f"Eğitim sohbet verisi: {len(train_data)} token")
print(f"Test sohbet verisi: {len(val_data)} token")

In [ ]:
n = int(0.9*len(data))#datayı parça parça koyuyoruz çünkü hepsini tek koyarsak onu ezberler ve sadece onu çıkarmaya başlar ama %80 bi yerde %20 bi yerde ezberlerlerse birbirinden farklı jenerasyonlar elde edeeriz bize dökümanın aynısını değil döküman "gibi" çıktılar üretir
train_data = data[:n]
val_data = data[n:]


In [31]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data)-block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size]for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    x,y =x.to(device),y.to(device)
    return x,y

In [ ]:
x = train_data[:block_size] #block size kadar bir alanı alırız
y = train_data[1:block_size+1]# x in 1 adım kaydırılmış halidirmodelin tahmin etmeye çalışacağı targetler burda tutulur

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("input",context,"iken targetimiz",target)

In [33]:
@torch.no_grad()
def estimate_loss():
    out ={}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X,Y = get_batch(split)
            logits,loss = model(X ,Y)
            losses[k] = loss.mean()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self,num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.proj._is_residual_proj = True
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
        self.net[2]._is_residual_proj = True

    def forward(self, x):
        return self.net(x)
class Block(nn.Module):
    def __init__(self,n_embd,n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x
        
class GPTLanguageModel(nn.Module): 
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.lm_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight

        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            if hasattr(module, '_is_residual_proj') and module._is_residual_proj:
               torch.nn.init.normal_(module.weight, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))
            else:
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        
    def forward(self, index, targets=None):
        B,T = index.shape

        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.lm_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape 
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
        
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=50, eos_token=None):
        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            
            logits, loss = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
            
            if eos_token is not None and index_next.item() == eos_token:
                break
        
        return index

In [ ]:
model = GPTLanguageModel(vocab_size)
model.load_state_dict(torch.load('ai_save.pt', map_location=device))
model = model.to(device)

In [ ]:
model.load_state_dict(torch.load('ai_save_alpaca.pt', map_location=device))
model = model.to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iters, eta_min=1e-6)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        current_lr = scheduler.get_last_lr()[0]
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}, lr: {current_lr:.2e}")
        torch.save(model.state_dict(), save_path)
    xb,yb = get_batch('train')
    
    #kaybı hesapla
    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()
print(loss.item())

In [ ]:
context = torch.zeros((1,1), dtype=torch.long, device = device)
generated_chars= decode(model.generate(context, max_new_tokens=500, temperature=0.8, top_k=50)[0].tolist())
print(generated_chars)

In [ ]:
import torch
import tiktoken

model.eval()
enc = tiktoken.get_encoding("gpt2")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

sorum = "Develop a plan to reduce electricity usage in a home." 

formatted_prompt = f"### Talimat:\n{sorum}\n\n### Cevap:\n"

context_tokens = enc.encode(formatted_prompt)
context = torch.tensor([context_tokens], dtype=torch.long, device=device)
eos_id = enc.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

with torch.no_grad():
    generated_tokens = model.generate(context, max_new_tokens=200, temperature=0.8, top_k=50, eos_token=eos_id)[0].tolist()

tam_cevap = enc.decode(generated_tokens)

asistan_cevabi = tam_cevap.split("### Cevap:\n")[-1]
asistan_cevabi = asistan_cevabi.replace("<|endoftext|>", "").strip()

print("Asistan: " + asistan_cevabi)